Introduce packages

In [ ]:
## a useful function to generate a data list for further analysis
import os,json,sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
from scipy.io import savemat
from analyse_stimulus_evoked_response import main
##need to add this additional cell because useful tools are in another folder. Need to integrate these two folders one day
current_working_directory = Path.cwd()
parent_dir = current_working_directory.resolve().parents[0]
sys.path.insert(0, str(parent_dir) + "\\utilities")
from useful_tools import select_animals_gpt, find_file
from data_cleaning import preprocess_fictrac_data

Session 0.1: Load analysis methods in python dictionary form

In [ ]:
sheet_name="MatrexVR"
#sheet_name="VCCball"
#sheet_name="Zball"
json_file = "./analysis_methods_dictionary.json"
with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())
    

Datasets="Z:/DATA/experiment_trackball_Optomotor"
this_dataset_dir = f"{Datasets}/{sheet_name}"

In [ ]:
##introduce some helper class to define colour gradient
import matplotlib as mpl
from matplotlib import cm
class MplColorHelper:
    def __init__(self, cmap_name, start_val, stop_val):
        self.cmap_name = cmap_name
        self.cmap = plt.get_cmap(cmap_name)
        self.norm = mpl.colors.Normalize(vmin=start_val, vmax=stop_val)
        self.scalarMap = cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

    def get_rgb(self, val):
        return self.scalarMap.to_rgba(val)
colormap_name = "viridis"
# colormap_name = "coolwarm"
# COL = MplColorHelper(colormap_name, 0, len(df["stim_type"].unique()))
# sm = cm.ScalarMappable(cmap=colormap_name)

Session 2.5: Load existing dataset

In [ ]:
#database_pattern = "matrexVR_coherence_database.pickle"
database_pattern = "matrex_vr_coherence_database_yaw_ori.pickle"
load_existing_dataset=True
this_database = find_file(this_dataset_dir, database_pattern)
velocity_pattern="velocity*.npy"
jaw_vector_pattern="*jaw_rotation.npy"
summary_pattern="behavioural*.pickle"
pattern_list=[velocity_pattern,jaw_vector_pattern,summary_pattern]
print(f"loading this {this_database}")
if this_database is None or this_database.is_file()==False:
    print("no post-analysis database created. Either run analysis cell to get the list of animal's response or assemble post-analysis dataset from individual animals")
    if len(summary_list)>0:
    #if 'summary_list' in locals():
        df=pd.DataFrame(pd.concat(summary_list))
        print("use local list created from jupyter notebook")
    else:
        velocity_list=[]
        summary_list=[]
        rotation_list=[]
        for this_pattern in pattern_list:
            for this_dir in dir_list:
                this_file = find_file(this_dir, this_pattern)
                if this_file.suffix==".npy":
                    print({f"npy file is found. Let's assemble them"})
                    if this_pattern==velocity_pattern:
                        velocity_list.append(np.load(this_file))
                    elif this_pattern==jaw_vector_pattern:
                        rotation_list.append(np.load(this_file))
                elif this_file.suffix==".pickle":
                    print({f"load individual animal's summary"})
                    summary_list.append(pd.read_pickle(this_file))
                else:
                    print({f"no particular file created in this path: {this_dir}. Skip this animal"})
                

        df_velocity=np.concatenate(velocity_list)
        df_rotation=np.concatenate(rotation_list)
        df=pd.DataFrame(pd.concat(summary_list,axis=0, ignore_index=True))
elif load_existing_dataset==True:
    df=pd.read_pickle(this_database)
    df.rename(columns = {'Unnamed: 0':'trial_number'}, inplace = True)
    print("load pickle into pandas dataframe")
else:
    print("do not know what to do here")

split database based on if animals are active or not; and then analyse turn dependent or independent from direction

In [ ]:
def converting_follow_velocity(row):
    if row['stim_type'] <0:
        row['stim_type'] *= -1    # Check condition on the first column
        row.loc['ori_f0':] *= -1        # Modify the second column (columnA)
    return row


In [ ]:
df_filtered = df[~df['ID'].isin([3,7,15,16,19,35,38,65])]
df_interest_no_lazy= df_filtered.apply(converting_follow_velocity, axis=1)
df_interest = df.apply(converting_follow_velocity, axis=1)
df_filtered.reset_index(inplace=True)
df_interest_no_lazy.reset_index(inplace=True)

In [ ]:
#define the first and last frames to analyse
stim_duration=analysis_methods.get("stim_duration")
interval_duration=analysis_methods.get("interval_duration")
if type(interval_duration) == list and type(stim_duration) == list:
    analysis_window = [
        min(interval_duration) * -1,
        max(stim_duration) + min(interval_duration),
    ]
elif type(interval_duration) == list:
    analysis_window = [
        min(interval_duration) * -1,
        stim_duration + min(interval_duration),
    ]
elif type(stim_duration) == list:
    analysis_window = [
        interval_duration * -1,
        max(stim_duration) + interval_duration,
    ]
else:
    analysis_window = [
        interval_duration * -1,
        stim_duration + interval_duration,
    ]
init_frame=analysis_methods.get("camera_fps")*abs(analysis_window[0])
end_frame=analysis_methods.get("camera_fps")*analysis_window[1]
print(init_frame,end_frame)

In [ ]:
def plot_yaw_angular_velocity(df,analysis_methods,init_frame=1000,end_frame=3500,vr_no="all",colormap_name="coolwarm"):
    save_output= analysis_methods.get("save_output",False)
    visual_paradigm_name= analysis_methods.get("experiment_name")
    camera_fps=analysis_methods.get("camera_fps")
    active_trial_only=analysis_methods.get("active_trial_only",False)
    COL = MplColorHelper(colormap_name, 0, len(df["stim_type"].unique())-1)
    sm = cm.ScalarMappable(cmap=colormap_name)
    using_bs_ci=False
    use_mean=True
    stimulus_velocity=64 #unit degree/s
    walk_threshold_tbt=25
    premature_turn=0.25#set this criteria to remove animals starting to make turn turing ISI
    fig, axes = plt.subplots(
        nrows=1, ncols=2, figsize=(12, 10))
    ax1, ax2, = axes.flatten()

    fig_1by1, subplots = plt.subplots(
        nrows=1, ncols=len(df["stim_type"].unique())+1, figsize=(20, 4), tight_layout=True
    )
    if active_trial_only==True:
        df_selected=df[(df['travel_distance_isi']>walk_threshold_tbt) & (abs(df["opto_index_isi"])<premature_turn)]
    else:
        df_selected=df
    i=0
    for this_stim, entries in df_selected.groupby('stim_type'):
        this_w=entries.loc[:,'ori_f0':].values
        this_dif_w=np.diff(this_w,axis=1)
        time_points = np.arange(this_w.shape[1])
        time_points_b = np.tile(time_points, (this_w.shape[0], 1))
        #define the stimulus speed in the first iteration
        if i ==0:
            xticks = [0, init_frame, end_frame]
            xticklabels = [str(analysis_window[0]), "0", str(stim_duration)]
            y_range=150
            tmp = np.full(this_w.shape[1],np.nan)
            dif_tmp= np.full(this_dif_w.shape[1],np.nan)
            stim_w_array=np.ones(end_frame-init_frame)*stimulus_velocity*np.pi/180/camera_fps
            tmp[init_frame:end_frame]=np.cumsum(stim_w_array)
            dif_tmp[init_frame:end_frame]=stim_w_array
        #ax1.plot(time_points_b, this_w, linestyle='-',linewidth=0.1,color=COL.get_rgb(i))

        #plot trial-by-trial response
        ax1.scatter(time_points_b, this_w, s=0.01,alpha=0.1,color=COL.get_rgb(i))
        subplots[i].scatter(time_points_b, this_w, s=0.01,alpha=0.1,color=COL.get_rgb(i))
        subplots[i].plot(time_points,tmp,color='k',linewidth=0.8)
        subplots[i].plot(time_points,tmp*-1,color='k',linewidth=0.8)
        subplots[i].axvline(
        x=init_frame,
        color="black",
        linestyle="--",
        label="Stimulus Starts",
        )
        subplots[i].axvline(
            x=end_frame,
            color="black",
            linestyle="--",
            label="Stimulus Ends",
        )
        subplots[i].set(
        ylabel="Orientation (rad)",
        yticks=[-y_range,0,y_range],
        ylim=[-y_range,y_range],
        xticks=xticks,
        xticklabels=xticklabels,
        xlabel="Time (s)",
    )
        #plot average response
        if use_mean==True:
            Ws = np.mean(this_w, axis=0)
            diff_Ws = np.mean(this_dif_w, axis=0)
        else:
            Ws=np.median(this_w, axis=0)
            diff_Ws=np.median(this_dif_w, axis=0)
        #ax2.plot(time_points,median_response,color=COL.get_rgb(i))
        ax2.plot(time_points[1:],diff_Ws,color=COL.get_rgb(i))
        subplots[len(df["stim_type"].unique())].plot(time_points,Ws,color=COL.get_rgb(i))

        ##to plot distribution with bootstrap 95% confidence interval
        if using_bs_ci:
            # data=(this_w,)
            # res=st.bootstrap(data,np.std,n_resamples=99,vectorized=True,random_state=1)
            # y1=res.confidence_interval[0]
            # y2=res.confidence_interval[1]
            # del data
            # data=(this_dif_w,)
            # res=st.bootstrap(data,np.std,n_resamples=99,vectorized=True,random_state=1)
            # dif_y1=res.confidence_interval[0]
            # dif_y2=res.confidence_interval[1]
            confidence_level = 0.95
            ##to plot distribution with 95% confidence interval with normal distribution
            cl95=st.norm.interval(confidence_level,loc=Ws,scale=st.sem(this_w))
            y1=cl95[0][:]
            y2=cl95[1][:]
            cl95=st.norm.interval(confidence_level,loc=diff_Ws,scale=st.sem(this_dif_w))
            dif_y1=cl95[0][:]
            dif_y2=cl95[1][:]

        else:
            ##to plot distribution with sem
            sem_response = np.std(this_w, axis=0, ddof=1) / np.sqrt(this_w.shape[0])
            y1 = Ws + sem_response
            y2 = Ws - sem_response
            sem_response = np.std(this_dif_w, axis=0, ddof=1) / np.sqrt(this_dif_w.shape[0])
            dif_y1=diff_Ws + sem_response
            dif_y2=diff_Ws - sem_response
        #ax2.fill_between(time_points,y1,y2, alpha=0.5,color=COL.get_rgb(i))
        ax2.fill_between(time_points[1:],dif_y1,dif_y2, alpha=0.5,color=COL.get_rgb(i))
        subplots[len(df["stim_type"].unique())].fill_between(time_points,y1,y2, alpha=0.5,color=COL.get_rgb(i))

        i=i+1

    ax1.plot(time_points,tmp,color='k',linewidth=0.8)
    ax1.plot(time_points,tmp*-1,color='k',linewidth=0.8)
    ax2.plot(time_points[1:],dif_tmp,color='k',linewidth=0.8)
    ax2.plot(time_points[1:],dif_tmp*-1,color='k',linewidth=0.8)
    subplots[len(df["stim_type"].unique())].plot(time_points,tmp,color='k',linewidth=0.8)
    subplots[len(df["stim_type"].unique())].plot(time_points,tmp*-1,color='k',linewidth=0.8)
    ax1.axvline(
        x=init_frame,
        color="black",
        linestyle="--",
        label="Stimulus Starts",
    )
    ax1.axvline(
        x=end_frame,
        color="black",
        linestyle="--",
        label="Stimulus Ends",
    )
    ax2.axvline(
        x=init_frame,
        color="black",
        linestyle="--",
        label="Stimulus Starts",
    )
    ax2.axvline(
        x=end_frame,
        color="black",
        linestyle="--",
        label="Stimulus Ends",
    )
    subplots[len(df["stim_type"].unique())].axvline(
        x=init_frame,
        color="black",
        linestyle="--",
        label="Stimulus Starts",
    )
    subplots[len(df["stim_type"].unique())].axvline(
        x=end_frame,
        color="black",
        linestyle="--",
        label="Stimulus Ends",
    )
    ax1.set(
        ylabel="Orientation (rad)",
        yticks=[-y_range,0,y_range],
        ylim=[-y_range,y_range],
        xticks=xticks,
        xticklabels=xticklabels,
        xlabel="Time (s)",
    )
    ax2.set(
        ylabel="Angular velocity (rad/s)",
        xticks=xticks,
        yticks=[-0.01,0,0.01],
        #ylim=[-0.005,0.02],
        ylim=[-0.025,0.025],
        xticklabels=xticklabels,
        xlabel="Time (s)",
    )
    subplots[len(df["stim_type"].unique())].set(
        xticks=xticks,
        yticks=[-20,0,20],
        xticklabels=xticklabels,
        xlabel="Time (s)",
    )
    tick_positions = np.linspace(0, 1, len(df["stim_type"].unique()))
    these_coherence=df["stim_type"].unique()
    these_coherence.sort()
    tick_labels = [str(value) for value in these_coherence]
    cbar = plt.colorbar(sm, ax=[ax1,ax2], orientation="horizontal", location='top',ticks=tick_positions,aspect=80)
    cbar.set_ticklabels(tick_labels)

    if save_output==True:
        fig_name=f"{vr_no}_bonfic_{visual_paradigm_name}_orientation_dynamics.png"
        fig.savefig(Path(this_dataset_dir)/fig_name)
        fig_name=f"{vr_no}_bonfic_{visual_paradigm_name}_orientation_dynamics_tbt.png"
        fig_1by1.savefig(Path(this_dataset_dir)/fig_name)

sort trials or ISI with the same length

In [ ]:

def split_trials_with_ISI(df):
    #trial_of_interest=df['preStim_ISI_duration']==10
    trial_of_interest=df['previous_trial_duration']==10
    ## select trials whose post-stim ISI is 10 sec
    #those_index=np.where(trial_of_interest.values)[0]+1
    those_index=np.where(trial_of_interest.values)[0]
    if df.shape[0]==max(those_index):
        df_interest=df.iloc[those_index[:-1],:]
    else:
        df_interest=df.iloc[those_index,:]
    return df_interest

In [ ]:
df_15_no_dir=split_trials_with_ISI(df_interest)
df_interest_no_lazy.reset_index(inplace=True)
df_active_15_no_dir=split_trials_with_ISI(df_interest_no_lazy)

In [ ]:
df_10=split_trials_with_ISI(df)
df_active_10=split_trials_with_ISI(df_filtered)

analyse response across time

In [ ]:
def response_across_time(df,analysis_methods):
    (init_frame,end_frame)=(1000,3500)
    colormap_name = "viridis"
    time_binned_analysis=3
    if time_binned_analysis==3:
        condition_of_interests=[df['stim_index']<max(df['stim_index'].values)/3,(df['stim_index']>max(df['stim_index'].values)/3) & (df['stim_index']<2*max(df['stim_index'].values)/3),df['stim_index']>2*max(df['stim_index'].values)/3]
    for k in range(len(condition_of_interests)):
        df_interest=df[condition_of_interests[k]]
        plot_yaw_angular_velocity(df_interest,analysis_methods,init_frame,end_frame,"all",colormap_name)

In [ ]:
response_across_time(df_interest,analysis_methods)

analysis travel distance across time

In [ ]:
condition_of_interests=[df['stim_index']<max(df['stim_index'].values)/3,(df['stim_index']>max(df['stim_index'].values)/3) & (df['stim_index']<2*max(df['stim_index'].values)/3),df['stim_index']>2*max(df['stim_index'].values)/3]
num_conditions=len(condition_of_interests)
num_animals=len(df_interest['ID'].unique())
these_stim_walk_distance=np.ones((num_conditions,num_animals))
these_isi_walk_distance=np.ones((num_conditions,num_animals))
for k in range(num_conditions):
    this_condition=condition_of_interests[k]
    df_interest=df[this_condition]
    for j in range(num_animals):
        this_id=df_interest['ID'].unique()[j]
        entries=df_interest[df_interest['ID']==this_id]
        these_stim_walk_distance[k,j]=entries['travel_distance'].sum()
        these_isi_walk_distance[k,j]=entries['travel_distance_isi'].sum()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 14), tight_layout=True)
ax1, ax2, ax3= axes.flatten()
ax1.plot(these_stim_walk_distance)
ax1.plot(np.mean(these_stim_walk_distance,axis=1),'k',linewidth=5)
ax1.plot(np.median(these_stim_walk_distance,axis=1),'k',linewidth=5,alpha=0.6)
ax1.set_ylim(0, 100000)
ax1.set(
    xticks=[0,1.0,2.0],
    xticklabels=['T1','T2','T3'],
    ylabel="Travel distance (mm)",
    xlabel="Time",
    title='stim')
ax2.plot(these_isi_walk_distance)
ax2.plot(np.mean(these_isi_walk_distance,axis=1),'k',linewidth=5)
ax2.plot(np.median(these_isi_walk_distance,axis=1),'k',linewidth=5,alpha=0.6)
ax2.set_ylim(0, 20000)
ax2.set(
    xticks=[0,1.0,2.0],
    xticklabels=['T1','T2','T3'],
    ylabel="Travel distance (mm)",
    xlabel="Time",
    title='5-sec prior to stim')
ax3.plot(these_stim_walk_distance/these_isi_walk_distance)
ax3.plot(np.mean(these_stim_walk_distance/these_isi_walk_distance,axis=1),'k',linewidth=5)
ax3.plot(np.median(these_stim_walk_distance/these_isi_walk_distance,axis=1),'k',linewidth=5,alpha=0.6)
ax3.set_ylim(0, 10)
ax3.set(
    xticks=[0,1.0,2.0],
    xticklabels=['T1','T2','T3'],
    ylabel="Ratio of travel distance",
    xlabel="Time")

analyse visual or motor experiences

In [ ]:
#analyse experience-dependent effect, 4 conditions, 100 coherence, animals turn, 100 coherence, animal dont turn, and then angular velocity in the following trials
def turn_coherence_analysis(df,analysis_methods):
    coherence_focused=False
    turn_threshold=0.25
    for this_coherence in [-100,-66,-33,0,33,66,100]:
        print(this_coherence)
        if coherence_focused==True:
            if this_coherence>0:
                df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']>turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
            else:
                df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']<-1*turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
        else:
            if this_coherence>0:
                df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                #df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']<-1*turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                #df_interest=df.iloc[df[(df['stim_type']>50) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
            else:
                df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                #df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']<-1*turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                #df_interest=df.iloc[df[(df['stim_type']<-50) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
        final=df_interest
        #final=df_interest[df_interest['previous_trial_duration']==10]
        plot_yaw_angular_velocity(final,analysis_methods)

In [ ]:
#analyse experience-dependent effect, 4 conditions, 100 coherence, animals turn, 100 coherence, animal dont turn, and then angular velocity in the following trials
def turn_coherence_analysis(df,analysis_methods):
    coherence_focused=True
    turn_threshold=0.25
    for this_coherence in [-50,-33,0,33,50]:
        print(this_coherence)
        if coherence_focused==True:
            if this_coherence>0:
                if this_coherence==33:
                    df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']>turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                else:
                    df_interest=df.iloc[df[(df['stim_type']>this_coherence) & (df['opto_index']>turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
            else:
                if this_coherence==-33 or this_coherence==0:
                    df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']<-1*turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                else:
                    df_interest=df.iloc[df[(df['stim_type']<this_coherence) & (df['opto_index']<-1*turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
        else:
            if this_coherence>0:
                df_interest=df.iloc[df[(df['stim_type']>=this_coherence) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                #df_interest=df.iloc[df[(df['stim_type']>50) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
            else:
                df_interest=df.iloc[df[(df['stim_type']<=-1*this_coherence) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
                #df_interest=df.iloc[df[(df['stim_type']<-50) & (abs(df['opto_index'])<turn_threshold)&(df["stim_index"]%335!=0)].index+1,:]
        final=df_interest
        #final=df_interest[df_interest['previous_trial_duration']==10]
        plot_yaw_angular_velocity(final,analysis_methods)

In [ ]:
turn_coherence_analysis(df,analysis_methods)

In [ ]:
for this_coherence in [0,33,66,100]:
    if this_coherence>0:
        df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']>0.25)&(df["stim_index"]%335!=0)].index+1,:]
    else:
        df_interest=df.iloc[df[(df['stim_type']==this_coherence) & (df['opto_index']<-0.25)&(df["stim_index"]%335!=0)].index+1,:]
    plot_yaw_angular_velocity(df_interest,analysis_methods)